# Lab 5: Fulltext Search with Neo4jContextProvider

In this lab, you will configure `Neo4jContextProvider` with fulltext search for keyword-based matching using Neo4j's built-in fulltext indexing with BM25 ranking. Unlike vector search which finds conceptual matches, fulltext search finds exact word matches — ideal when users query with specific names, titles, or terms.

## What you will learn

- How to configure `Neo4jContextProvider` with `index_type="fulltext"`
- That fulltext search does not require an embedder
- How stop word filtering works
- When to choose fulltext vs vector search

## Setup

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

provider_name = os.getenv("LLM_PROVIDER", "openai").lower()
if provider_name == "azure":
    assert os.getenv("AZURE_OPENAI_API_KEY"), "AZURE_OPENAI_API_KEY not set in .env file"
    assert os.getenv("AZURE_OPENAI_ENDPOINT"), "AZURE_OPENAI_ENDPOINT not set in .env file"
else:
    assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not set in .env file"
assert os.getenv("NEO4J_URI"), "NEO4J_URI not set in .env file"
print("Environment loaded successfully")

## Import Dependencies

In [ ]:
from llm_provider import get_client
from agent_framework_neo4j import Neo4jContextProvider, Neo4jSettings

## Load Neo4j Settings

In [ ]:
neo4j_settings = Neo4jSettings()

## Create the Fulltext Provider

Fulltext search uses BM25 ranking to match exact keywords in the indexed text — it finds literal term matches rather than semantic similarity, making it ideal for queries with specific names or titles. Configure `Neo4jContextProvider` with:
- `index_name` set to `neo4j_settings.fulltext_index_name`
- `index_type="fulltext"`
- No embedder needed — fulltext search uses BM25 keyword matching
- `top_k=5`
- A `context_prompt`

In [ ]:
provider = Neo4jContextProvider(
    uri=neo4j_settings.uri,
    username=neo4j_settings.username,
    password=neo4j_settings.get_password(),
    index_name=neo4j_settings.fulltext_index_name,
    index_type="fulltext",
    top_k=5,
    context_prompt=(
        "## Knowledge Graph Context\n"
        "Use the following information from the knowledge graph "
        "to answer questions about movies, actors, and genres:"
    ),
)

## Run the Agent

Try a keyword-based query. Fulltext search excels with specific names and terms that should match literally in movie plots.

In [ ]:
async with provider:
    client = get_client()

    agent = client.as_agent(
        name="fulltext-agent",
        instructions=(
            "You are a helpful assistant that answers questions about "
            "movies using the provided knowledge graph context. "
            "Be concise and cite specific information from the context "
            "when available."
        ),
        context_providers=[provider],
    )

    session = agent.create_session()

    query = "Batman Gotham"
    print(f"User: {query}\n")
    print("Answer: ", end="", flush=True)
    response = await agent.run(query, session=session)
    print(response.text)
    print()

## Summary

Fulltext search with `Neo4jContextProvider` uses `index_type="fulltext"` to match queries against a Neo4j fulltext index using BM25 ranking. It doesn't require an embedder, filters stop words by default, and works best for keyword-based queries with specific names or terms.

**What's next:** In the next lab, you will combine vector and fulltext search using hybrid mode.